# Prompt and chunking variants on Kaggle (revision, 40 pages)

Review items M4 (prompt variants) and M3 (model-independent chunks), for the four
general-purpose models, both OCR engines, plain greedy decoding. Variants, in
priority order: preserve-spelling prompt; 300-character chunks; Bengali prompt;
three-shot prompt (examples from development pages). The original prompt with
token chunks is already in `gpu_eval40.jsonl`.

**Before running:**
1. Add data: the dataset with `ocr_eval40.jsonl` **and** `fewshot_examples.json`. To resume a
   stopped run, also add that run's output (it holds a partial `gpu_variants40.jsonl`).
2. Session options: Accelerator = GPU T4 x2, Internet = on. Secret `HF_TOKEN` attached.
3. Save Version, then "Save & Run All (Commit)".

When it finishes, copy `gpu_variants40.jsonl` from the version's Output tab.

In [ ]:
import glob, os, shutil, subprocess, time

# Kaggle background runs stop at 12 h; start no new page after 11 h.
DEADLINE = time.time() + 11 * 3600

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT = "/kaggle/working/gpu_variants40.jsonl"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Ariffurrhmn/bengali-postocr-sllm.git", REPO_DIR], check=True)
subprocess.run("pip install -q -U transformers accelerate sentencepiece jiwer", shell=True, check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

OCR = find("ocr_eval40.jsonl")
FEWSHOT = find("fewshot_examples.json")
assert OCR and FEWSHOT, "Attach the dataset with ocr_eval40.jsonl and fewshot_examples.json"
shutil.copy(FEWSHOT, f"{REPO_DIR}/data/fewshot_examples.json")
print("OCR pages:", sum(1 for _ in open(OCR, encoding="utf-8")))

previous = find("gpu_variants40.jsonl")
if previous and not os.path.exists(OUT):
    shutil.copy(previous, OUT)
    print("Resuming from", previous)
print("Already done:", sum(1 for _ in open(OUT, encoding="utf-8")) if os.path.exists(OUT) else 0)

In [ ]:
MODELS = "gemma-2b,qwen3-1.7b,phi4-mini,llama3.2-1b"

def sweep(prompt, approach):
    cmd = ["python", "run_sweep.py", "--split", "eval", "--models", MODELS,
           "--engines", "tesseract,easyocr", "--approaches", approach,
           "--decoding", "plain", "--prompt", prompt,
           "--ocr-path", OCR, "--out", OUT,
           "--stop-after-minutes", str(max(0, (DEADLINE - time.time()) / 60))]
    with open("/kaggle/working/sweep.log", "a", encoding="utf-8") as log:
        subprocess.run(cmd, cwd=f"{REPO_DIR}/correction", stdout=log, stderr=log)
    subprocess.run(r"grep -E '^\[|FAILED|Error' /kaggle/working/sweep.log | tail -3", shell=True)

sweep("en_preserve", "chunked")

In [ ]:
sweep("en", "chunked_chars")

In [ ]:
sweep("bn", "chunked")

In [ ]:
sweep("fewshot", "chunked")

In [ ]:
import collections, json
rows = [json.loads(l) for l in open(OUT, encoding="utf-8") if l.strip()]
count = collections.Counter((r["model_key"], r["prompt"], r["approach"]) for r in rows)
for key in sorted(count):
    print(key, count[key])
print("total", len(rows), "of", 4 * 4 * 80, "| errors:", sum(1 for r in rows if r.get("error")))